In [2]:
import pandas as pd 
import numpy as np
from xgboost import XGBClassifier,plot_importance
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,accuracy_score,confusion_matrix
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE


In [3]:
df=pd.read_csv("HR_Employee_Attrition.csv")

In [4]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [6]:
for i in df.columns:
    unique=df[i].unique()
    if len(unique)==1:
        df.drop(columns=[i],inplace=True)
    elif len(unique)==len(df["Age"]):
        df.drop(columns=[i],inplace=True)
#It removes all columns where all rows contain a single value or where each row has a unique value.


In [7]:
df.dropna(inplace=True)

In [8]:
lr=LabelEncoder()

In [9]:
for i in df.columns:
    if df[i].dtype ==object:
        df[i]=lr.fit_transform(df[i])

In [10]:
x=df.drop(columns=["Attrition"])
y=df["Attrition"]

In [11]:
smote=SMOTE()
x1,y1=smote.fit_resample(x,y)

In [12]:
x1

,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,2,1102,2,1,2,1,2,0,94,...,3,1,0,8,0,1,6,4,0,5
1,49,1,279,1,8,1,1,3,1,61,...,4,4,1,10,3,3,10,7,1,7
2,37,2,1373,1,2,2,4,4,1,92,...,3,2,0,7,3,3,0,0,0,0
3,33,1,1392,1,3,4,1,4,0,56,...,3,3,0,8,3,3,8,7,3,0
4,27,2,591,1,2,1,3,1,1,40,...,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2461,50,1,541,1,2,4,4,2,0,78,...,3,2,1,31,4,3,30,7,2,8
2462,24,2,833,1,3,3,3,1,1,60,...,3,1,0,5,1,2,2,0,2,0
2463,28,1,1293,0,22,3,0,1,0,56,...,3,1,0,2,1,2,1,0,0,0
2464,26,2,1426,1,16,3,3,1,1,46,...,3,3,0,4,2,2,3,2,0,1


In [13]:
xgb=XGBClassifier()
xgb.fit(x1,y1)
importance=xgb.feature_importances_
features=x.columns

indices = np.argsort(importance)[::-1]  
sorted_features = features[indices]
sorted_importances = importance[indices]
sorted_x=sorted_features[:10][::-1]
x2=x1[sorted_x]
x2

,OverTime,RelationshipSatisfaction,BusinessTravel,EnvironmentSatisfaction,PerformanceRating,JobSatisfaction,JobLevel,JobInvolvement,MaritalStatus,StockOptionLevel
0,1,1,2,2,3,4,2,3,2,0
1,0,4,1,3,4,2,2,2,1,1
2,1,2,2,4,3,3,1,2,2,0
3,1,3,1,4,3,3,1,3,1,0
4,0,4,2,1,3,2,1,3,1,1
...,...,...,...,...,...,...,...,...,...,...
2461,0,2,1,2,3,4,3,2,0,1
2462,1,1,2,1,3,3,1,2,1,0
2463,0,1,1,1,3,2,1,2,0,0
2464,1,3,2,1,3,2,1,2,0,0


In [14]:
x_train,x_test,y_train,y_test=train_test_split(x2,y1,test_size=0.2)

In [15]:
line=LogisticRegression()
line.fit(x_train,y_train)

LogisticRegression()

In [16]:
pred_line_r=line.predict(x_train)
print("According to training data R2 Scorec :",r2_score(y_train,pred_line_r))

According to training data R2 Scorec : 0.20483870967741957


In [17]:
pred_line=line.predict(x_test)
print("According to testing data R2 Score :",r2_score(y_test,pred_line))

According to testing data R2 Score : 0.17360143014120988


In [18]:
dt=DecisionTreeClassifier(max_depth=3)
dt.fit(x_train,y_train)

DecisionTreeClassifier(max_depth=3)

In [19]:
pred_decision_r=dt.predict(x_train)
print("According to training data Accuracy Score :",accuracy_score(y_train,pred_decision_r))

According to training data Accuracy Score : 0.7682555780933062


In [20]:
pred_decision=dt.predict(x_test)
print("According to testing data Accuracy Score :",accuracy_score(y_test,pred_decision))

According to testing data Accuracy Score : 0.7348178137651822


In [21]:
xgb=XGBClassifier(max_depth=1,n_estimators=300,learning_rate=0.3,subsample=0.8,gamma=0.1)
xgb.fit(x_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=0.1, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.3, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=1, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [22]:
pred_xgb_train=xgb.predict(x_train)
print("According to training data Accuracy Score :",accuracy_score(y_train,pred_xgb_train))


According to training data Accuracy Score : 0.829107505070994


In [23]:
pred_xgb_test=xgb.predict(x_test)
print("According to testing data Accuracy Score :",accuracy_score(y_test,pred_xgb_test))

According to testing data Accuracy Score : 0.8259109311740891


In [24]:
print(confusion_matrix(y_test,pred_xgb_test))

[[215  38]
 [ 48 193]]


In [25]:
x_train

,OverTime,RelationshipSatisfaction,BusinessTravel,EnvironmentSatisfaction,PerformanceRating,JobSatisfaction,JobLevel,JobInvolvement,MaritalStatus,StockOptionLevel
1320,0,2,0,3,4,3,1,3,1,2
450,1,3,2,2,3,4,2,3,2,0
2219,0,1,2,2,3,1,2,2,0,2
1032,1,2,0,1,4,1,1,2,2,0
2168,0,3,1,3,3,2,1,2,2,0
...,...,...,...,...,...,...,...,...,...,...
1914,0,2,1,3,3,1,1,3,2,0
790,0,3,2,4,3,4,3,2,0,1
204,1,2,2,2,3,1,2,3,1,0
1857,0,3,1,2,3,2,2,2,1,1


In [28]:
def prediction():
    StockOptionLevel=int(input("Enter Stock Option Level (0-3) : "))
    JobInvolvement=int(input("Enter Job Involvement (1-4) : "))
    JobLevel=int(input("Enter Job level (1-5) : "))
    JobSatisfaction=int(input("Enter Job Satisfaction (1-4) : "))
    MaritalStatus=int(input("Enter Marital Status (Divorced:-0, Married:-1, Single:-2) : "))
    BusinessTravel=int(input("Enter Business Travel ( Non-Travel:-0, Travel_Frequently:-1 Travel_Rarely:-2) :"))
    WorkLifeBalance=int(input("Enter Work Life Balance (1-4) : "))
    EnvironmentSatisfaction=int(input("Enter Environment Satisfaction (1-4) : "))
    RelationshipSatisfaction=int(input("Enter Relationship Satisfaction (1-4) : "))
    PerformanceRating=int(input("Enter Performance Rating (3-4) : "))
    
    x=[[StockOptionLevel, JobInvolvement, JobLevel,
        JobSatisfaction, MaritalStatus, BusinessTravel,
        WorkLifeBalance,EnvironmentSatisfaction, 
        RelationshipSatisfaction, PerformanceRating 
        ]]
    final_result=xgb.predict(x)
    return final_result
result=prediction()


In [29]:
if result==0:
    print("He/She can work for few years")
else:
    print("He/she can leave anytime ")

He/She can work for few years
